In [17]:
from __future__ import annotations

import os
import sys
import argparse
from pathlib import Path
from typing import Any, Iterable

import json
import random
import wandb
import numpy as np
from collections import Counter, defaultdict
from dataclasses import dataclass
from PIL import Image, ImageDraw
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models


# io_utils.py

In [2]:
def ensure_parent_dir(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)

def write_json(path: Path, payload: dict[str, Any]) -> None:
    ensure_parent_dir(path)
    with path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)

def write_jsonl(path: Path, rows: Iterable[dict[str, Any]]) -> None:
    ensure_parent_dir(path)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=True) + "\n")

def read_jsonl(path: Path) -> List[dict[str, Any]]:
    if not path.exists():
        raise FileNotFoundError(f"JSONL not found: {path}")
    rows: list[dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows

# dataset.py

In [3]:
def default_transform(image_size: int) -> transforms.Compose:
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

def _apply_face_mask(image: Image.Image, face_bbox: Any) -> Image.Image:
    if not face_bbox:
        return image.copy()
    
    if not isinstance(face_bbox, (list, tuple)) or len(face_bbox) != 4:
        return image.copy()
    
    x1, y1, x2, y2 = [int(v) for v in face_bbox]
    masked = image.copy()
    draw = ImageDraw.Draw(masked)
    draw.rectangle([x1, y1, x2, y2], fill=(127, 127, 127))
    return masked

class CAERSTwoStreamDataset(Dataset):
    def __init__(
        self,
        manifest_path: Path,
        dataset_root: Path,
        split: str,
        image_size: int,
    ) -> None:
        self.dataset_root = dataset_root
        self.split = split
        self.transform = default_transform(image_size)

        rows = read_jsonl(manifest_path)
        self.rows = [r for r in rows if str(r.get("split")) == split]
        if len(self.rows) == 0:
            raise ValueError(f"No rows found for split={split} in manifest={manifest_path}")
        
        labels = sorted({str(r["label"]) for r in rows})
        self.label_to_index = {label: idx for idx, label in enumerate(labels)}
        self.index_to_label = {idx: label for label, idx in self.label_to_idx.items()}

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, index: int) -> dict[str, Any]:
        row = self.rows[index]
        rel_path = str(row["image_path"])
        image_path = self.dataset_root / rel_path
        if not image_path.exists():
            raise FileNotFoundError(f"Missing image file: {image_path}")
        
        image = Image.open(image_path).convert("RGB")
        context_image = _apply_face_mask(image, row.get("face_bbox"))

        face_tensor = self.transform(image)
        context_tensor = self.transform(context_image)

        label_name = str(row["label"])
        label_idx = self.label_to_index[label_name]

        return {
            "sample_id": str(row["sample_id"]),
            "face_image": face_tensor,
            "context_image": context_tensor,
            "label": torch.tensor(label_idx, dtype=torch.long),
            "label_name": label_name,
            "image_path": rel_path,
        }

# data_manifest.py

In [4]:
@dataclass
class ManifestBuildResult:
    rows: list[dict[str, object]]
    diagnostics: dict[str, object]

def _collect_split_rows(
    dataset_root: Path,
    split_name: str,
    image_extensions: tuple[str, ...],
) -> list[dict[str, object]]:
    split_dir = dataset_root / split_name
    if not split_dir.exists():
        return []
    
    rows: list[dict[str, object]] = []
    for class_dir in sorted(split_dir.iterdir()):
        if not class_dir.is_dir():
            continue
        label = class_dir.name

        for image_path in sorted(class_dir.rglob("*")):
            if not image_path.is_file():
                continue
            if image_path.suffix.lower() not in image_extensions:
                continue

            rel_path = image_path.relative_to(dataset_root).as_posix()
            sample_id = rel_path.replace("/", "__").replace(".", "_")
            rows.append({
                "sample_id": f"{split_name}__{sample_id}",
                "image_path": rel_path,
                "label": label,
                "split": split_name,
                "face_bbox": None,
            })
    return rows

def _stratified_holdout(
    rows: list[dict[str, object]],
    holdout_ratio: float,
    seed: int,
) -> tuple[list[dict[str, object]], list[dict[str, object]]]:
    by_label: dict[str, list[dict[str, object]]] = defaultdict(list)
    for row in rows:
        by_label[str(row["label"])].append(row)

    rnd = random.Random(seed)
    keep_rows: list[dict[str, object]] = []
    holdout_rows: list[dict[str, object]] = []

    for label, label_rows in by_label.items():
        pool = list(label_rows)
        rnd.shuffle(pool)
        holdout_n = max(1, int(round(len(pool) * holdout_ratio)))
        holdout_n = min(holdout_n, len(pool) - 1) if len(pool) > 1 else 0
        holdout = pool[:holdout_n]
        keep = pool[holdout_n:]

        for item in holdout:
            copied = dict(item)
            copied["split"] = "val"
            copied["sample_id"] = str(copied["sample_id"]).replace("train__", "val__", 1)
            holdout_rows.append(copied)

        keep_rows.extend(keep)

    return keep_rows, holdout_rows

def _split_counts(rows: list[dict[str, object]]) -> dict[str, int]:
    counter: Counter[str] = Counter()
    for row in rows:
        counter[str(row["split"])] = counter[str(row["split"])] + 1
    return dict(counter)

def _class_counts(rows: list[dict[str, object]], split_name: str) -> dict[str, int]:
    counter: Counter[str] = Counter()
    for row in rows:
        if str(row["split"]) == split_name:
            counter[str(row["label"])] = counter[str(row["label"])] + 1
    return dict(sorted(counter.items()))

def build_caers_manifest(
    dataset_root: Path,
    image_extensions: tuple[str, ...],
    create_val_split: bool,
    val_ratio: float,
    seed: int,
) -> ManifestBuildResult:
    train_rows = _collect_split_rows(dataset_root, "train", image_extensions)
    test_rows = _collect_split_rows(dataset_root, "test", image_extensions)

    if len(train_rows) == 0 and len(test_rows) == 0:
        raise FileNotFoundError(
            "Missing CAER-S split data. Expected to find at least one of 'train' or 'test' directories under dataset root: "
        )
    
    if create_val_split:
        train_rows, val_rows = _stratified_holdout(train_rows, val_ratio, seed)
    else:
        val_rows = []

    all_rows = train_rows + val_rows + test_rows

    diagnostics = {
        "dataset_root": str(dataset_root),
        "total_samples": len(all_rows),
        "split_counts": _split_counts(all_rows),
        "class_counts": {
            "train": _class_counts(all_rows, "train"),
            "val": _class_counts(all_rows, "val"),
            "test": _class_counts(all_rows, "test"),
        },
        "labels": sorted({str(r["label"]) for r in all_rows}),
    }

    return ManifestBuildResult(rows=all_rows, diagnostics=diagnostics)


# engine.py


In [5]:
def accuracy_topk(
    logits: torch.Tensor,
    targets: torch.Tensor,
    topk: tuple[int, ...] = (1,),
) -> list[float]:
    maxk = max(topk)
    batch_size = targets.size(0)
    _, pred = logits.topk(maxk, dim=1, largest=True, sorted=True)
    pred = pred.t()
    correct = pred.eq(targets.view(1, -1).expand_as(pred))
    res: list[float] = []
    for k in topk:
        correct_k = correct[:k].reshape(-1).float().sum(0, keepdim=True)
        res.append(float(correct_k.mul_(100.0 / batch_size).item()))
    return res

class MetricTracker:
    def __init__(self) -> None:
        self.reset()

    def reset(self) -> None:
        self.total_loss = 0.0
        self.total_samples = 0
        self.total_correct_top1 = 0
        self.total_correct_top5 = 0

    def update(self, loss: float, logtis: torch.Tensor, labels: torch.Tensor) -> None:
        batch_size = labels.size(0)
        self.total_loss += loss * batch_size
        self.total_samples += batch_size
        acc1, acc5 = accuracy_topk(logtis, labels, topk=(1, 5))
        self.total_correct_top1 += acc1 * batch_size
        self.total_correct_top5 += acc5 * batch_size

    @property
    def avg_loss(self) -> float:
        return self.total_loss / max(1, self.total_samples)
    
    @property
    def avg_acc1(self) -> float:
        return self.total_correct_top1 / max(1, self.total_samples) * 100.0
    
    @property
    def avg_acc5(self) -> float:
        return self.total_correct_top5 / max(1, self.total_samples) * 100.0
    
    def summary(self) -> dict[str, float]:
        return {
            "avg_loss": self.avg_loss,
            "avg_acc1": self.avg_acc1,
            "avg_acc5": self.avg_acc5,
        }
    
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader[Any],
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device: torch.device,
) -> dict[str, float]:
    model.train()
    tracker = MetricTracker()

    pbar = tqdm(loader, desc="train", leave=False)
    for batch in pbar:
        face = batch["face_image"].to(device)
        context = batch["context_image"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        out = model(face, context)
        loss = criterion(out["logits"], labels)
        loss.backward()
        optimizer.step()

        tracker.update(loss.item(), out["logits"].detach(), labels)
        pbar.set_postfix({"loss": f"{tracker.avg_loss:.4f}", "acc1": f"{tracker.avg_acc1:.2f}%"})

    return tracker.summary()

@torch.no_grad()
def evaluate_per_class(
    model: nn.Module,
    loader: DataLoader[Any],
    device: torch.device,
    index_to_label: dict[int, str],
) -> dict[str, Any]:
    model.eval()
    num_classes = len(index_to_label)
    correct_per_class = torch.zeros(num_classes, dtype=torch.long)
    total_per_class = torch.zeros(num_classes, dtype=torch.long)

    for batch in tqdm(loader, desc="eval_per_class", leave=False):
        face = batch["face_image"].to(device)
        context = batch["context_image"].to(device)
        labels = batch["label"].to(device)

        out = model(face, context)
        preds = out["logits"].argmax(dim=1)

        for c in range(num_classes):
            mask = labels == c
            if mask.any():
                total_per_class[c] += mask.sum().item()
                correct_per_class[c] += (preds[mask] == labels[mask]).sum().item()

    per_class_acc: dict[str, float] = {}
    for idx, label_name in index_to_label.items():
        total = int(total_per_class[idx])
        correct = int(correct_per_class[idx])
        per_class_acc[label_name] = (correct / total * 100.0) if total > 0 else 0.0

    overall_acc = float(correct_per_class.sum().item() / max(1, total_per_class.sum().item()) * 100.0)

    return {
        "overall_acc": overall_acc,
        "per_class_acc": per_class_acc,
    }



# model.py


In [6]:
def _make_encoder(backbone: str = "resnet18", pretrained: bool = False) -> tuple[nn.Module, int]:
    if backbone == "resnet18":
        weight = models.ResNet18_Weights.DEFAULT if pretrained else None
        net = models.resnet18(weights=weight)
        dim = 512
        net.fc = nn.Identity()
        return net, dim
    if backbone == "mobilenet_v2":
        weight = models.MobileNet_V2_Weights.DEFAULT if pretrained else None
        net = models.mobilenet_v2(weights=weight)
        dim = 1280
        net.classifier = nn.Identity()
        return net, dim
    raise ValueError(f"Unsupported backbone: {backbone}")

class AdaptiveFusion(nn.Module):
    def __init__(self, face_dim: int, context_dim: int, hidden_dim: int = 256) -> None:
        super().__init__()
        total_dim = face_dim + context_dim
        self.fc1 = nn.Linear(total_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 2)

    def forward(self, face_feat: torch.Tensor, context_feat: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        combined = torch.cat([face_feat, context_feat], dim=1)
        h = F.relu(self.fc1(combined))
        logits = self.fc2(h)
        weights = F.softmax(logits, dim=1)
        w_face = weights[:, 0:1]
        w_context = weights[:, 1:2]
        fused = w_face * face_feat + w_context * context_feat
        return fused, weights
    
class CAERNet(nn.Module):
    def __init__(
        self,
        num_classes: int,
        backbone: str = "resnet18",
        pretrained: bool = False,
        dropout: float = 0.5,
    ) -> None:
        super().__init__()
        self.face_encoder, self.face_dim = _make_encoder(backbone, pretrained=pretrained)
        self.context_encoder, self.context_dim = _make_encoder(backbone, pretrained=pretrained)

        self.fusion = AdaptiveFusion(self.face_dim, self.context_dim)

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.face_dim, num_classes),
        )
    
    def forward(
        self,
        face_image: torch.Tensor,
        context_image: torch.Tensor,
    ) -> dict[str, Any]:
        face_feat = self.face_encoder(face_image)
        context_feat = self.context_encoder(context_image)

        if face_feat.dim() > 2:
            face_feat = F.adaptive_avg_pool2d(face_feat, (1, 1)).flatten(1)
        if context_feat.dim() > 2:
            context_feat = F.adaptive_avg_pool2d(context_feat, (1, 1)).flatten(1)

        fused_feat, fusion_weights = self.fusion(face_feat, context_feat)
        logits = self.classifier(fused_feat)

        return {
            "logits": logits,
            "fusion_weights": fusion_weights,
            "face_feat": face_feat,
            "context_feat": context_feat,
            "fused_feat": fused_feat,
        }
    
class SingleStreamNet(nn.Module):
    def __init__(
        self,
        num_classes: int,
        stream: str = "face",
        backbone: str = "resnet18",
        pretrained: bool = False,
        dropout: float = 0.5,
    ) -> None:
        super().__init__()
        if stream not in ("face", "context"):
            raise ValueError(f"Unsupported stream: {stream}")
        self.stream = stream
        self.encoder, self.feat_dim = _make_encoder(backbone, pretrained=pretrained)
        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(self.feat_dim, num_classes),
        )

    def forward(self, face_image: torch.Tensor, context_image: torch.Tensor) -> dict[str, Any]:
        x = face_image if self.stream == "face" else context_image
        feat = self.encoder(x)
        if feat.dim() > 2:
            feat = F.adaptive_avg_pool2d(feat, (1, 1)).flatten(1)
        logits = self.classifier(feat)
        return {
            "logits": logits,
            "feat": feat,
        }

# train_caers.py

In [16]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def build_model(num_classes: int, cfg: object) -> nn.Module:
    if cfg.train.stream_mode == "multimodal":
        return CAERNet(
            num_classes=num_classes,
            backbone=cfg.model.backbone,
            pretrained=cfg.model.pretained,
            dropout=cfg.model.dropout,
        )
    return SingleStreamNet(
        num_classes=num_classes,
        stream=cfg.train.stream_mode,
        backbone=cfg.model.backbone,
        pretrained=cfg.model.pretained,
        dropout=cfg.model.dropout,
    )


In [13]:
from types import SimpleNamespace
from pathlib import Path

def dict_to_obj(d):
    if isinstance(d, dict):
        return SimpleNamespace(**{k: dict_to_obj(v) for k, v in d.items()})
    elif isinstance(d, list):
        return [dict_to_obj(i) for i in d]
    else:
        return d

In [ ]:
cfg = {
    "seed": 42,

    "dataset": {
        "dataset_root": "/home/agung/riset/risetv2/CAER-S",
        "image_extensions": [".png", ".jpg", ".jpeg", ".bmp", ".webp"],
        "create_val_split": True,
        "val_ratio": 0.1,
        "image_size": 224,
    },

    "model": {
        "backbone": "resnet18",
        "pretrained": True,
        "dropout": 0.5,
    },

    "train": {
        "batch_size": 32,
        "num_epochs": 30,
        "lr": 0.001,
        "weight_decay": 0.0001,
        "num_workers": 4,
        "device": "cuda",
        "stream_mode": "multimodal",
        "save_dir": "checkpoints/caers",
    },

    "outputs": {
        "manifest_path": "artifacts/caers/manifest_caers.jsonl",
        "diagnostics_path": "artifacts/caers/diagnostics_caers.json",
    },

    "wandb_api_key": "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",
    "wandb_offline": False,
    "wandb_project": "caers-emotion-recognition,",
    "wandb_entity": "Tim-1",
    "wandb_run_name": "CAER-Dual Stream-$(date +%Y%m%d-%H%M%S)",
}

cfg = dict_to_obj(cfg)

In [ ]:
set_seed(42)
device = torch.device("cuda")

if cfg.wandb_api_key and not cfg.wandb_offline:
    wandb.login(key=cfg.wandb_api_key)

wandb_mode = "offline" if cfg.wandb_offline else "online"
run = wandb.init(
    project=cfg.wandb_project,
    entity=cfg.wandb_entity,
    name=cfg.wandb_run_name,
    config=vars(cfg),
)

ds_train = CAERSTwoStreamDataset(
    manifest_path=cfg.outputs.manifest_path,
    dataset_root=cfg.dataset.dataset_root,
    split="train",
    image_size=cfg.dataset.image_size,
)

ds_val = CAERSTwoStreamDataset(
    manifest_path=cfg.outputs.manifest_path,
    dataset_root=cfg.dataset.dataset_root,
    split="val",
    image_size=cfg.dataset.image_size,
)

loader_train = DataLoader(
    ds_train,
    batch_size=cfg.train.batch_size,
    shuffle=True,
    num_workers=cfg.train.num_workers,
    pin_memory=True,
)

loader_val = DataLoader(
    ds_val,
    batch_size=cfg.train.batch_size,
    shuffle=False,
    num_workers=cfg.train.num_workers,
    pin_memory=True,
)

num_classes = len(ds_train.label_to_index)
model = build_model(num_classes, cfg).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.train.lr, weight_decay=cfg.train.weight_decay)

wandb.watch(model, log="all", log_freq=100)

start_epoch = 0
best_val_acc = 0.0
history: list[dict[str, object]] = []

resume = "/checkpoints/caers/best_model.pth"

if resume:
    checkpoint = torch.load(resume, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    start_epoch = checkpoint.get("epoch", 0) + 1
    best_val_acc = checkpoint.get("best_val_acc", 0.0)
    print(f"Resumed from epoch {start_epoch}, best_val_acc={best_val_acc:.2f}%")
    wandb.run.summary["resumed_from_epoch"] = start_epoch
    wandb.run.summary["resumed_best_val_acc"] = best_val_acc

ensure_parent_dir(cfg.train_save_dir / "placeholder")

for epoch in range(start_epoch, cfg.train.num_epochs):
    print(f"Epoch {epoch+1}/{cfg.train.num_epochs}")
    train_metrics = train_one_epoch(model, loader_train, optimizer, criterion, device)
    val_metrics = evaluate_per_class(model, loader_val, criterion, device)

    print(f" Train: loss={train_metrics['loss']:.4f}, acc1={train_metrics['acc1']:.2f}%, acc5={train_metrics['acc5']:.2f}%")
    print(f" Val: loss={val_metrics['loss']:.4f}, acc1={val_metrics['acc1']:.2f}%, acc5={val_metrics['acc5']:.2f}%")

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_metrics["loss"],
        "train_acc1": train_metrics["acc1"],
        "train_acc5": train_metrics["acc5"],
        "val_loss": val_metrics["loss"],
        "val_acc1": val_metrics["acc1"],
        "val_acc5": val_metrics["acc5"],
    })

    history.append({
        "epoch": epoch + 1,
        "train": train_metrics,
        "val": val_metrics,
    })

    if val_metrics["acc1"] > best_val_acc:
        best_vall_acc = val_metrics["acc1"]
        checkpoint_path = cfg.train.save_dir / "best_model.pth"
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_acc": best_val_acc,
        }, checkpoint_path)
        print(f" Saved best model -> {checkpoint_path}")

        artifact = wandb.Artifact(
            name=f"caers-model-{wandb.run.id}",
            type="model",
            metadata={
                "epoch": epoch + 1,
                "val_acc1": best_val_acc,
                "stream_mode": cfg.train.stream_mode,
            },
        )
        artifact.add_file(str(checkpoint_path))
        wandb.log_artifact(artifact, aliases=["best"])

history_path = cfg.train_save_dir / "history.json"
write_json(history_path, {"history": history, "best_val_acc": best_val_acc})
print(f"Training complete. History saved to {history_path}")

wandb.run.summary["best_val_acc"] = best_val_acc
wandb.run.summary["total_epochs"] = cfg.train.num_epochs
wandb.save(str(history_path))

wandb.finish()

42